In [2]:
import os, glob, duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import average_precision_score, roc_auc_score

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW  = '../data/raw/epoch_data'
PROC = '../data/processed'

D          = '2026-09-01'
TRAIN_DATE = '2026-08-11'     # 공식 train_labels (FUT=18 고정, 그대로 사용)
HOLD_DATE  = '2026-08-25'     # 공식 train_labels (FUT=18 고정, 그대로 사용)
FUT_FINAL  = 26               # 본선 채점 기준. 우리가 직접 만드는 추가 기준일에 적용

con = duckdb.connect()
con.sql("SET memory_limit = '4GB'")
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")
con.sql(f"CREATE OR REPLACE VIEW video_snapshots AS SELECT * FROM '{PROC}/video_snapshots.parquet'")
print('준비 완료')

준비 완료


In [3]:
def make_features(Dref):
    """기준일 Dref 시점 피처. CUT_HOURS 제한 없음 (09/01 이후 관측값도 past 영상이면 사용 — 공지로 확인됨)"""
    return con.sql(f"""
    WITH v5 AS (
        SELECT channel_id,
               COUNT(*) AS n_long_5d,
               MEDIAN(view_5d) AS past_med,
               MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS recent14_med,
               MEDIAN(like_5d / NULLIF(view_5d, 0)) AS like_rate,
               STDDEV(LN(1 + view_5d)) AS log_view_std
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL
          AND published_at < TIMESTAMP '{Dref}'          -- CUT_HOURS 제거, published_at만 기준
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0 AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id,
               COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN up USING (channel_id) LEFT JOIN games USING (channel_id)
    """).df()

def make_labels(Dref, FUT=FUT_FINAL):
    """라벨 재현 (이제 FUT=26 기본값). CUT_HOURS 없이 published_at만으로 past/future 구분 (공지로 확인됨)"""
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (
                     WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {FUT} DAY), 0) AS future_score
          FROM video_5d_views
          WHERE is_shorts = 0 AND view_5d IS NOT NULL
          GROUP BY 1
        ),
        scored AS (
          SELECT channel_id, LN(1+future_score)-LN(1+past_score) AS growth
          FROM agg WHERE n_past >= 3 AND past_score >= 100
        )
        SELECT channel_id,
               CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored
    """).df()
    g['ref_date'] = Dref
    return g

In [4]:
EXTRA_DATES = ['2026-07-18', '2026-07-21', '2026-07-24', '2026-07-27', '2026-07-30', '2026-08-02', '2026-08-05']
extra_labels = pd.concat([make_labels(d) for d in EXTRA_DATES], ignore_index=True)
print(extra_labels.groupby('ref_date')['target'].agg(['size', 'mean']))

            size      mean
ref_date                  
2026-07-18   139  0.201439
2026-07-21   159  0.201258
2026-07-24   234  0.200855
2026-07-27   389  0.200514
2026-07-30   448  0.200893
2026-08-02   509  0.200393
2026-08-05   537  0.201117


In [5]:
# ===== 우리가 만든 FUT=26 라벨에 피처 붙이기 =====
extra_pieces = [extra_labels[extra_labels.ref_date == d].merge(make_features(d), on='channel_id', how='left')
                for d in EXTRA_DATES]
extra_f = pd.concat(extra_pieces, ignore_index=True)

# ===== 공식 train_labels (08-11, 08-25, FUT=18 고정) 불러와서 피처 붙이기 =====
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl['row_id'].str.rsplit('_', n=1).str[0]
tl['ref_date']   = tl['row_id'].str.rsplit('_', n=1).str[1]
tl = tl.rename(columns={'target': 'target'})   # 컬럼명 통일 확인용

official_f = pd.concat([
    tl[tl.ref_date == d].merge(make_features(d), on='channel_id', how='left')
    for d in [TRAIN_DATE, HOLD_DATE]
], ignore_index=True)

# ===== 전체 합치기 =====
all_data = pd.concat([official_f, extra_f], ignore_index=True)
print('전체 행 수:', len(all_data))
print(all_data.groupby('ref_date')['target'].agg(['size', 'mean']))
print()
print('결측 확인:')
print(all_data[['past_med','recent14_med','n_games']].isna().mean().round(3))

전체 행 수: 3682
            size      mean
ref_date                  
2026-07-18   139  0.201439
2026-07-21   159  0.201258
2026-07-24   234  0.200855
2026-07-27   389  0.200514
2026-07-30   448  0.200893
2026-08-02   509  0.200393
2026-08-05   537  0.201117
2026-08-11   601  0.201331
2026-08-25   666  0.201201

결측 확인:
past_med        0.000
recent14_med    0.014
n_games         0.000
dtype: float64
